# Team pressure analysis
Barca (team 0) vs Atletico (team 1) — Barca-Atletico first half

Covers: proximity pressure, passing-lane pressure, combined team pressure events, pressure by pitch zone, defensive line height, and time-to-press after losing the ball.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import paths

FPS = 25
PITCH_LENGTH = 105
PITCH_WIDTH = 68

player_frame_table = pd.read_parquet(paths.PLAYER_FRAME_TABLE_CACHE_PATH)
ball_frame_table = pd.read_parquet(paths.BALL_FRAME_TABLE_CACHE_PATH)
pass_events = pd.read_parquet(paths.PASS_EVENTS_PATH)  # confirm exact attribute name in paths.py


## 1. Proximity pressure — distance from each off-ball defender to the ball carrier

In [ ]:
PRESSURE_DIST_THRESH_M = 5.0

carrier_pos = ball_frame_table[
    ["frame_idx", "carrier_track_id", "carrier_team", "possession_team", "ball_pitch_x", "ball_pitch_y"]
]

pft = player_frame_table.merge(carrier_pos, on="frame_idx", how="left")

off_ball = pft[
    (pft["role"] == "player") &
    pft["carrier_track_id"].notna() &
    (pft["track_id"] != pft["carrier_track_id"]) &
    (pft["team"] != pft["carrier_team"])
].copy()

off_ball["dist_to_carrier"] = (
    (off_ball["pitch_x"] - off_ball["ball_pitch_x"]) ** 2 +
    (off_ball["pitch_y"] - off_ball["ball_pitch_y"]) ** 2
) ** 0.5

off_ball_pressure = off_ball.dropna(subset=["dist_to_carrier"]).copy()
off_ball_pressure["is_pressuring"] = off_ball_pressure["dist_to_carrier"] < PRESSURE_DIST_THRESH_M

team_pressure_per_frame = (
    off_ball_pressure.groupby(["frame_idx", "team", "carrier_team", "possession_team"], as_index=False)
    .agg(
        n_pressing=("is_pressuring", "sum"),
        n_defenders_on_pitch=("track_id", "nunique"),
        min_dist_to_carrier=("dist_to_carrier", "min"),
    )
)
team_pressure_per_frame["pressing_intensity"] = (
    team_pressure_per_frame["n_pressing"] / team_pressure_per_frame["n_defenders_on_pitch"]
)


## 2. Passing-lane pressure — how covered are the carrier's passing options

In [ ]:
MIN_LANE_LENGTH_M = 3.0
TIGHT_LANE_THRESH_M = 1.5
CONTESTED_LANE_THRESH_M = 3.0

carrier_frames = pft[(pft["role"] == "player") & pft["carrier_track_id"].notna()].copy()

teammates = carrier_frames[
    (carrier_frames["team"] == carrier_frames["carrier_team"]) &
    (carrier_frames["track_id"] != carrier_frames["carrier_track_id"])
]
opponents = carrier_frames[carrier_frames["team"] != carrier_frames["carrier_team"]]

carrier_player_pos = (
    player_frame_table[player_frame_table["is_carrier"] == True]
    [["frame_idx", "track_id", "pitch_x", "pitch_y"]]
    .rename(columns={"track_id": "carrier_track_id", "pitch_x": "carrier_x", "pitch_y": "carrier_y"})
    .drop_duplicates(subset="frame_idx", keep="first")
)

teammate_pos = teammates[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "teammate_id", "pitch_x": "tm_x", "pitch_y": "tm_y"}
)
opponent_pos = opponents[["frame_idx", "track_id", "pitch_x", "pitch_y"]].rename(
    columns={"track_id": "opponent_id", "pitch_x": "opp_x", "pitch_y": "opp_y"}
)

lanes = teammate_pos.merge(opponent_pos, on="frame_idx", how="inner")
lanes = lanes.merge(carrier_player_pos, on="frame_idx", how="inner")
lanes = lanes.dropna(subset=["carrier_x", "carrier_y"])

Ax, Ay = lanes["carrier_x"].values, lanes["carrier_y"].values
Bx, By = lanes["tm_x"].values, lanes["tm_y"].values
Px, Py = lanes["opp_x"].values, lanes["opp_y"].values

ABx, ABy = Bx - Ax, By - Ay
APx, APy = Px - Ax, Py - Ay

seg_len_sq = ABx ** 2 + ABy ** 2
seg_len_sq = np.where(seg_len_sq == 0, 1e-9, seg_len_sq)

t = (APx * ABx + APy * ABy) / seg_len_sq
t_clipped = np.clip(t, 0.0, 1.0)

closest_x = Ax + t_clipped * ABx
closest_y = Ay + t_clipped * ABy
perp_dist = np.sqrt((Px - closest_x) ** 2 + (Py - closest_y) ** 2)

lanes["t"] = t
lanes["lane_perp_dist"] = perp_dist
lanes["lane_length_m"] = np.sqrt(seg_len_sq)

valid_lanes = lanes[
    (lanes["lane_length_m"] >= MIN_LANE_LENGTH_M) &
    (lanes["t"] >= 0.15) &
    (lanes["t"] <= 0.85)
].copy()

valid_lanes["lane_tight_block"] = valid_lanes["lane_perp_dist"] < TIGHT_LANE_THRESH_M
valid_lanes["lane_contested"] = valid_lanes["lane_perp_dist"] < CONTESTED_LANE_THRESH_M

lane_status = (
    valid_lanes.groupby(["frame_idx", "teammate_id"], as_index=False)
    .agg(lane_tight_block=("lane_tight_block", "any"), lane_contested=("lane_contested", "any"))
)

frame_lane_summary = (
    lane_status.groupby("frame_idx", as_index=False)
    .agg(
        n_lanes_total=("teammate_id", "nunique"),
        n_lanes_tight_blocked=("lane_tight_block", "sum"),
        n_lanes_contested=("lane_contested", "sum"),
    )
)
frame_lane_summary["n_lanes_open"] = frame_lane_summary["n_lanes_total"] - frame_lane_summary["n_lanes_contested"]
frame_lane_summary["frac_tight_blocked"] = frame_lane_summary["n_lanes_tight_blocked"] / frame_lane_summary["n_lanes_total"]
frame_lane_summary["frac_contested"] = frame_lane_summary["n_lanes_contested"] / frame_lane_summary["n_lanes_total"]

frame_lane_summary = frame_lane_summary.merge(
    carrier_pos[["frame_idx", "carrier_team", "possession_team"]], on="frame_idx", how="left"
)


## 3. Unified team pressure signal + events (proximity OR lane-blocking, min duration, gap-tolerant)

In [ ]:
N_PRESSING_THRESH = 2
FRAC_TIGHT_BLOCKED_THRESH = 0.375
MIN_EVENT_DURATION_SEC = 0.5
GAP_TOLERANCE_FRAMES = 5

team_pressure_full = team_pressure_per_frame.merge(
    frame_lane_summary[["frame_idx", "n_lanes_total", "n_lanes_tight_blocked",
                         "n_lanes_contested", "frac_tight_blocked", "frac_contested"]],
    on="frame_idx",
    how="left"
)

tpf = team_pressure_full.copy()
tpf["frac_tight_blocked"] = tpf["frac_tight_blocked"].fillna(0)
tpf["under_team_pressure"] = (
    (tpf["n_pressing"] >= N_PRESSING_THRESH) |
    (tpf["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)
)

tpf = tpf.sort_values(["team", "frame_idx"]).reset_index(drop=True)

events_list = []
for team_id, grp in tpf.groupby("team"):
    grp = grp.sort_values("frame_idx").reset_index(drop=True)

    is_press = grp["under_team_pressure"].values.copy()
    frames = grp["frame_idx"].values
    i, n = 0, len(is_press)
    while i < n:
        if not is_press[i]:
            j = i
            while j < n and not is_press[j]:
                j += 1
            gap_frame_span = frames[j - 1] - frames[i] + 1 if j > i else 0
            if i > 0 and j < n and gap_frame_span <= GAP_TOLERANCE_FRAMES:
                is_press[i:j] = True
            i = j
        else:
            i += 1
    grp["under_team_pressure_bridged"] = is_press

    grp["state_change"] = (grp["under_team_pressure_bridged"] != grp["under_team_pressure_bridged"].shift()).cumsum()
    runs = (
        grp.groupby("state_change")
        .agg(
            start_frame=("frame_idx", "first"),
            end_frame=("frame_idx", "last"),
            n_frames=("frame_idx", "count"),
            under_pressure=("under_team_pressure_bridged", "first"),
            possession_team=("possession_team", "first"),
            mean_n_pressing=("n_pressing", "mean"),
            mean_frac_tight_blocked=("frac_tight_blocked", "mean"),
        )
        .reset_index(drop=True)
    )
    runs["team"] = team_id
    events_list.append(runs)

team_pressure_events = pd.concat(events_list, ignore_index=True)
team_pressure_events = team_pressure_events[team_pressure_events["under_pressure"]].copy()
team_pressure_events["duration_sec"] = team_pressure_events["n_frames"] / FPS
team_pressure_events = team_pressure_events[
    team_pressure_events["duration_sec"] >= MIN_EVENT_DURATION_SEC
].reset_index(drop=True)

print("total team pressure events:", len(team_pressure_events))
print(team_pressure_events.groupby("team")["duration_sec"].describe())


## 4. Total pressure time, normalized by each team's out-of-possession window

In [ ]:
total_pressure_frames_by_team = team_pressure_events.groupby("team")["n_frames"].sum()
total_pressure_seconds_by_team = total_pressure_frames_by_team / FPS

possession_frames_by_team = ball_frame_table["possession_team"].value_counts(dropna=True)
total_labeled_frames = ball_frame_table["possession_team"].notna().sum()
out_of_possession_frames = {
    0: total_labeled_frames - possession_frames_by_team.get(0, 0),
    1: total_labeled_frames - possession_frames_by_team.get(1, 0),
}
out_of_possession_seconds = {k: v / FPS for k, v in out_of_possession_frames.items()}

match_duration_sec = (player_frame_table["frame_idx"].max() + 1) / FPS

print("total pressure seconds by team:\n", total_pressure_seconds_by_team)
print(f"\nmatch duration: {match_duration_sec:.1f}s")
for team_id in [0, 1]:
    pct = total_pressure_seconds_by_team.get(team_id, 0) / out_of_possession_seconds[team_id] * 100
    print(f"team {team_id}: {total_pressure_seconds_by_team.get(team_id, 0):.1f}s pressing / "
          f"{out_of_possession_seconds[team_id]:.1f}s out of possession = {pct:.1f}% pressing intensity")


## 5. Pressure by pitch zone (own / middle / attacking third, relative to pressing team's attack direction)

In [ ]:
ball_x_by_frame = ball_frame_table[["frame_idx", "ball_pitch_x"]].rename(columns={"ball_pitch_x": "press_x"})
tpf_zone = team_pressure_full.merge(ball_x_by_frame, on="frame_idx", how="left")

def classify_zone(row):
    x = row["press_x"]
    if pd.isna(x):
        return None
    if row["team"] == 0:
        if x < 35: return "own_third"
        elif x < 70: return "middle_third"
        else: return "attacking_third"
    else:
        if x > 70: return "own_third"
        elif x > 35: return "middle_third"
        else: return "attacking_third"

tpf_zone["press_zone"] = tpf_zone.apply(classify_zone, axis=1)
tpf_zone["frac_tight_blocked"] = tpf_zone["frac_tight_blocked"].fillna(0)
tpf_zone["under_team_pressure"] = (
    (tpf_zone["n_pressing"] >= N_PRESSING_THRESH) | (tpf_zone["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)
)

pressing_only = tpf_zone[tpf_zone["under_team_pressure"] & tpf_zone["press_zone"].notna()]

zone_counts = pressing_only.groupby(["team", "press_zone"]).size().unstack(fill_value=0)
zone_pct = zone_counts.div(zone_counts.sum(axis=1), axis=0) * 100

print("pressure frames by team and zone:\n", zone_counts)
print("\n% distribution within each team's own pressing:\n", zone_pct)


## 6. Defensive line height (distance from own goal, last-1 and last-3 defenders)

In [ ]:
BOUNDS_TOLERANCE_M = 2.0

outfield = player_frame_table[player_frame_table["role"] == "player"].copy()
outfield = outfield.dropna(subset=["pitch_x", "pitch_y"])

in_bounds = (
    (outfield["pitch_x"] >= -BOUNDS_TOLERANCE_M) & (outfield["pitch_x"] <= PITCH_LENGTH + BOUNDS_TOLERANCE_M) &
    (outfield["pitch_y"] >= -BOUNDS_TOLERANCE_M) & (outfield["pitch_y"] <= PITCH_WIDTH + BOUNDS_TOLERANCE_M)
)
outfield = outfield[in_bounds].copy()

outfield["dist_from_own_goal"] = np.where(
    outfield["team"] == 0,
    outfield["pitch_x"],
    PITCH_LENGTH - outfield["pitch_x"]
)
outfield["dist_from_own_goal"] = outfield["dist_from_own_goal"].clip(0, PITCH_LENGTH)

def compute_line_height(df, n_defenders):
    return (
        df.sort_values("dist_from_own_goal")
        .groupby(["frame_idx", "team"])
        .head(n_defenders)
        .groupby(["frame_idx", "team"], as_index=False)["dist_from_own_goal"]
        .mean()
        .rename(columns={"dist_from_own_goal": "line_height_m"})
    )

def remove_outliers_iqr(df, col, group_cols):
    parts = []
    for _, g in df.groupby(group_cols):
        q1, q3 = g[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        parts.append(g[(g[col] >= lo) & (g[col] <= hi)])
    return pd.concat(parts, ignore_index=True)

poss = ball_frame_table[["frame_idx", "possession_team"]]

line_last1 = compute_line_height(outfield, 1).merge(poss, on="frame_idx", how="left")
line_last3 = compute_line_height(outfield, 3).merge(poss, on="frame_idx", how="left")

line_last1_clean = remove_outliers_iqr(line_last1, "line_height_m", ["team"])
line_last3_clean = remove_outliers_iqr(line_last3, "line_height_m", ["team"])

avg_line_height = line_last3_clean.groupby("team")["line_height_m"].mean()

print("last-1 defender:\n", line_last1_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]))
print("\nlast-3 defenders avg:\n", line_last3_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]))


## 7. Defensive line height — pitch visualization

In [ ]:
team0_line_x = avg_line_height[0]
team1_line_x = PITCH_LENGTH - avg_line_height[1]

fig, ax = plt.subplots(figsize=(12, 7.5))

ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, color="black", linewidth=1.2))
ax.axvline(PITCH_LENGTH / 2, color="black", linewidth=0.8)
ax.add_patch(patches.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, fill=False, color="black", linewidth=0.8))

box_depth, box_width = 16.5, 40.3
ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - box_width) / 2), box_depth, box_width, fill=False, color="black", linewidth=0.8))
ax.add_patch(patches.Rectangle((PITCH_LENGTH - box_depth, (PITCH_WIDTH - box_width) / 2), box_depth, box_width, fill=False, color="black", linewidth=0.8))

ax.axvline(team0_line_x, color="#1D9E75", linewidth=3, label=f"Barca line ({team0_line_x:.1f}m from own goal)")
ax.axvline(team1_line_x, color="#534AB7", linewidth=3, label=f"Atletico line ({avg_line_height[1]:.1f}m from own goal)")

ax.set_xlim(-5, PITCH_LENGTH + 5)
ax.set_ylim(-5, PITCH_WIDTH + 5)
ax.set_aspect("equal")
ax.axis("off")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=False, fontsize=11)
ax.set_title("Average defensive line height (last-3 defenders avg)", fontsize=13, pad=15)

plt.tight_layout()
plt.show()


## 8. Time-to-press after losing the ball
Turnovers derived from raw (unsmoothed) carrier-team changes, filtered to stable holds (>=1.5s) to exclude 50/50 scramble noise.

In [ ]:
MIN_HOLD_SEC = 1.5
MIN_HOLD_FRAMES = int(MIN_HOLD_SEC * FPS)
MAX_SEARCH_WINDOW_SEC = 10
MAX_SEARCH_WINDOW_FRAMES = MAX_SEARCH_WINDOW_SEC * FPS

raw_carrier = ball_frame_table.dropna(subset=["carrier_team"])[["frame_idx", "carrier_team"]].sort_values("frame_idx").reset_index(drop=True)
raw_carrier["state_change"] = (raw_carrier["carrier_team"] != raw_carrier["carrier_team"].shift()).cumsum()
carrier_runs = (
    raw_carrier.groupby("state_change")
    .agg(start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"), carrier_team=("carrier_team", "first"))
    .reset_index(drop=True)
)

stable_runs = carrier_runs[carrier_runs["n_frames"] >= MIN_HOLD_FRAMES].reset_index(drop=True)

turnovers = []
for i in range(len(stable_runs) - 1):
    losing_team = stable_runs.loc[i, "carrier_team"]
    gaining_team = stable_runs.loc[i + 1, "carrier_team"]
    if losing_team != gaining_team:
        turnovers.append({"losing_team": losing_team, "loss_frame": stable_runs.loc[i, "end_frame"]})
turnovers = pd.DataFrame(turnovers)

press_lookup = team_pressure_full.copy()
press_lookup["frac_tight_blocked"] = press_lookup["frac_tight_blocked"].fillna(0)
press_lookup["under_team_pressure"] = (
    (press_lookup["n_pressing"] >= N_PRESSING_THRESH) | (press_lookup["frac_tight_blocked"] >= FRAC_TIGHT_BLOCKED_THRESH)
)
press_lookup = press_lookup[["frame_idx", "team", "under_team_pressure"]]

results = []
for _, row in turnovers.iterrows():
    team_id = row["losing_team"]
    loss_frame = row["loss_frame"]
    window = press_lookup[
        (press_lookup["team"] == team_id) &
        (press_lookup["frame_idx"] >= loss_frame) &
        (press_lookup["frame_idx"] <= loss_frame + MAX_SEARCH_WINDOW_FRAMES) &
        (press_lookup["under_team_pressure"])
    ]
    if len(window) > 0:
        first_press_frame = window["frame_idx"].min()
        results.append({"team": team_id, "loss_frame": loss_frame, "time_to_press_sec": (first_press_frame - loss_frame) / FPS})

time_to_press = pd.DataFrame(results)

print("genuine turnover events:", len(turnovers))
print(f"turnovers with a press detected within {MAX_SEARCH_WINDOW_SEC}s: {len(time_to_press)} / {len(turnovers)}")
print("\ntime-to-press stats by team (seconds):")
print(time_to_press.groupby("team")["time_to_press_sec"].agg(["mean", "median", "min", "max", "count"]))


## 9. Zone-weighted pressing intensity
Attacking-third pressure counts more than own-third pressure (own=1x, middle=2x, attacking=3x), since winning the ball high up the pitch is tactically more valuable. Score is bounded 0-100% by normalizing against the theoretical max (every out-of-possession frame spent pressing in the attacking third).

In [ ]:
ZONE_WEIGHTS = {"own_third": 1, "middle_third": 2, "attacking_third": 3}

weighted = tpf_zone[tpf_zone["under_team_pressure"] & tpf_zone["press_zone"].notna()].copy()
weighted["zone_weight"] = weighted["press_zone"].map(ZONE_WEIGHTS)

weighted_pressure_frames = weighted.groupby("team")["zone_weight"].sum()
weighted_pressure_seconds = weighted_pressure_frames / FPS

print("weighted pressure seconds by team (own=1x, middle=2x, attacking=3x):\n", weighted_pressure_seconds)

print("\nzone-weighted pressing intensity, bounded 0-100%:")
for team_id in [0, 1]:
    theoretical_max_seconds = out_of_possession_seconds[team_id] * ZONE_WEIGHTS["attacking_third"]
    pct_of_max = weighted_pressure_seconds.get(team_id, 0) / theoretical_max_seconds * 100
    print(f"team {team_id}: {pct_of_max:.1f}% of theoretical max weighted pressing")


## 10. Pass success rate under pressure vs not, per team
Uses the real pass-event table (passer/receiver, outcome, is_turnover) instead of the coarse carrier-hold proxy. Pressure is checked in the 1s window leading up to each pass release, using the opposing team's `under_team_pressure` flag.

In [ ]:
PRESSURE_LOOKBACK_SEC = 1.0
PRESSURE_LOOKBACK_FRAMES = int(PRESSURE_LOOKBACK_SEC * FPS)

press_flags = press_lookup.set_index(["frame_idx", "team"])["under_team_pressure"]

def was_under_pressure(passer_team, release_frame):
    passer_team = int(passer_team)
    opponent = 1 - passer_team
    release_frame = int(release_frame)
    for f in range(max(0, release_frame - PRESSURE_LOOKBACK_FRAMES), release_frame + 1):
        if press_flags.get((f, opponent), False):
            return True
    return False

pass_events["under_pressure"] = pass_events.apply(
    lambda row: was_under_pressure(row["passer_team"], row["passer_end_frame"]), axis=1
)
pass_events["success"] = pass_events["outcome"] == "completed"

print("total passes:", len(pass_events))
print(pass_events.groupby(["passer_team", "under_pressure"]).size())

summary = (
    pass_events.groupby(["passer_team", "under_pressure"])["success"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "success_rate"})
)
summary["success_rate"] = (summary["success_rate"] * 100).round(1)
print("\npass success rate by team, under pressure vs not:\n", summary)

print("\ntotal passes attempted while under pressure, by team:")
print(pass_events[pass_events["under_pressure"]].groupby("passer_team").size())
